In [ ]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Exercise 1.5 p 26

#### Helpers
Beta function is defined as follows:
$$
\begin{align}
B(x, y) &= \int_0^1 \theta^{x-1}(1-\theta)^{y-1} \mathrm{d}\theta \\
&= \frac{\Gamma(x)\Gamma(y)}{\Gamma(x+y)}
\end{align}
$$

***

#### Part 1 - $X$ distribution 

##### Experimentally 
To find it we just need to plot histogram of our X values, because we randomly select $\theta_i \rightarrow X_i$, by taking histogram of $X$ we make it independent of prior $\theta_i$ value. 

##### Analytically
We just go through using beta function property
$$
\begin{align*}
\mathbb{P}[X] 
&= 
\int_0^1 \mathbb{P}[X|\theta]\mathbb{P}[\theta]\mathrm{d}\theta = \int_0^1 \binom{n}{x}\theta^x(1-\theta)^{n-x}\mathrm{d}\theta \\
&= \binom{n}{x} \cdot \frac{x!(n-x)!}{(x + n -x + 1)!} = \binom{n}{x} \cdot \frac{x!(n-x)!}{(n+1)!} \\
&= \frac{1}{n+1}
\end{align*}
$$
We obtain uniform distribution

***

#### Part 2 - Posterior

##### Find experimentally $\theta$ posterior 
We will use elimination method if following setting: 
- we set $f=\mathbb{P}[\theta| X]$
- we set $g-\mathbb{P}[\theta]$

We obtain $M$ by estimate:
$$
\frac{\mathbb{P}[X|\theta]}{\mathbb{P}[X]} \leq \frac{1}{\mathbb{P}[X]} =: M
$$ 

By putting everything in acceptation function we obtain:
$$
\frac{f(\theta)}{M\cdot g(\theta)} = \frac{\frac{\mathbb{P}[X|\theta]\cdot \mathbb{P}[\theta]}{\mathbb{P}[X]}}{\frac{1}{\mathbb{P}[X]}\cdot \mathbb{P}[\theta]} = \mathbb{P}[X|\theta]
$$
Then we accept our element if:
$$
U \leq \mathbb{P}[X|\theta] 
$$


> In lecture function are given $f=p$ and $g=q$

##### Find a posteriori distribution of $\theta$ parameter
To find a posteriori $\theta$ parameter we use Bayes theorem
$$
\begin{align*}
\mathbb{P}[\theta|X] 
&= 
\frac{\mathbb{P}[X|\theta]\mathbb{P}[\theta]}{\mathbb{P}[X]} 
\varpropto 
\mathbb{P}[X|\theta]\mathbb{P}[\theta] 
= 
\binom{n}{x}\theta^x(1-\theta)^{n-x} \cdot 1 \\
&\varpropto 
\theta^x(1-\theta)^{n-x}
\end{align*}
$$
We can conclude that this is kernel of beta distribution:
$$
\theta|X \sim \mathrm{\beta}(x+1, n-x + 1)
$$
from which be obtain constant, for $X=3, n=9$ we obtain
$$
\theta|X \sim \mathrm{\beta}(4, 7) 
$$

***

### Part 3 - different order

We now sample values in reverse order:
$$
\mathbb{P}[\theta] = \sum_x \mathbb{P}[\theta|X]\mathbb{P}[X]
$$

In [ ]:
# helpers
## params
sample_size = 1_000_000
param_n_binom = 9

## functions 
def cond_dist_rvs(n, param_n_binom=param_n_binom):
    r'''
    Function which return probe from hierarchical model
    
    $$\theta \sim \mathrm{Unif}(0, 1), X \sim \mathrm{Binom}(n, \theta)$$

    It return probe from $\mathrm{Binom}$ 
    '''
    thetas = stats.distributions.uniform().rvs(n) # (n, ) 
    x = np.apply_along_axis(
        lambda x: stats.distributions.binom(param_n_binom, x).rvs(),
        axis=0,
        arr=thetas
    )
    return np.concat((thetas.reshape(-1, 1), x.reshape(-1, 1)), axis=1)  # (n, 2)

def cond_dist_rvs_part3(n, param_n_binom=param_n_binom):
    X = stats.distributions.uniform(0, param_n_binom).rvs(n)
    thetas = np.apply_along_axis(
        lambda x: stats.distributions.beta(x+1,param_n_binom -x + 1).rvs(),
        axis=0,
        arr=X
    )
    return thetas

In [ ]:
# Part 1 - X distribution
## We obtain from it marginal distribution (we take random values and sum)
probe = cond_dist_rvs(sample_size)
domain_ex15_part1 = np.linspace(0, 9, 100)
## plots
sns.histplot(probe[:, 1], stat='density', bins=param_n_binom+1)
plt.plot(domain_ex15_part1, stats.distributions.uniform(0, param_n_binom).pdf(domain_ex15_part1), color='red')

In [ ]:
# Part 2 - analytical distribution
## params
domain_ex15_part2 = np.linspace(0, 1, 100)
X_3_array = probe[probe[:, 1] == 3]
## generalization - we would need go through all values x 

## obtain probabilities - (n,2) -> (m, ) 
probabilites = np.apply_along_axis(lambda x: stats.distributions.binom(param_n_binom, x[0]).pmf(x[1]), axis=1, arr=X_3_array)

## accept right \theta
### sample U (m, ) 
condition = stats.distributions.uniform().rvs(probabilites.shape[0])
### condition = condition =< probabilites
condition = condition <= probabilites 

## plots 
sns.histplot(X_3_array[condition][:, 0], stat="density")
plt.plot(domain_ex15_part2, 
         stats.distributions.beta(4, 7).pdf(domain_ex15_part2), 
         color='red')

In [ ]:
# part 3
domain_ex15_part3 = np.linspace(0, 1, 100)
thetas_3 = cond_dist_rvs_part3(100_000)
sns.histplot(thetas_3, stat='density')
plt.plot(domain_ex15_part3, stats.distributions.uniform(0, 1).pdf(domain_ex15_part3), color='red')

### Exercise 1.7 p 27 

#### Part 2 - Martingale check

##### Finite expected value
$$
M_n = \frac{\alpha + S_n}{\alpha + \beta + n} \leq \frac{\alpha + n}{\alpha + \beta + n} < 1
$$

<!-- Case for $X_1$
$$
\mathbb{E}[X_1] = \frac{\alpha}{\alpha+\beta}
$$
Case for $X_2$ (Induction for $n=1$)
$$
\mathbb{E}[X_2] = \frac{\alpha + \mathbb{E}[X_1]}{\alpha + \beta + 1} \leq \frac{\alpha + 1}{\alpha + \beta + 1}
$$

General equation
$$
\mathbb{E}[M_n] = \frac{\alpha + \mathbb{E}[S_{n}]}{\alpha + \beta + n} \leq \frac{\alpha + n}{\alpha + \beta + n}
$$
Induction proof:
$$
\begin{align*}
\frac{\alpha + \mathbb{E}[S_{n}]}{\alpha + \beta + n} 
&= 
\frac{ \alpha + \mathbb{E}[X_n] + \mathbb{E}[S_{n-1}]}{\alpha + \beta + n} \leq \\
&\leq 
\frac{ \alpha + \mathbb{E}[X_n] + n-1}{\alpha + \beta + n} = \frac{ \alpha + \frac{\alpha + \mathbb{E}[S_{n-1}]}{\alpha + \beta + n-1} + n-1}{\alpha + \beta + n} \leq \\
&\leq
\frac{ \alpha + \frac{\alpha + n-1}{\alpha + \beta + n-1} + n-1}{\alpha + \beta + n}
\leq \frac{ \alpha + 1 + n-1}{\alpha + \beta + n} = \\
&= 
\frac{ \alpha + n}{\alpha + \beta + n} < 1 
\end{align*} 
$$ -->

##### Martingale property

$$
\begin{align*}
\mathbb{E}\left[ M_{n+1} |\mathcal{F_{n}}\right] 
 &= \mathbb{E}\left[ \frac{\alpha + X_{n+1} + S_n}{\alpha + \beta + n+1} | \mathcal{F_n}\right] \\
 & = \frac{1}{\alpha+ \beta+ n+1}\mathbb{E}[\alpha+ X_{n+1}+S_{n}|\mathcal{F_{n}}] = \\
 & = \frac{1}{\alpha+\beta+n+1}(\alpha+S_{n}+\mathbb{P}[X_{n+1}|\mathcal{F}_{n}]\cdot1) \\
 & = \frac{1}{\alpha+\beta+n+1}\left( \alpha+S_{n} + \frac{{\alpha+S_{n}}}{\alpha+\beta+n} \right) =\\
 & = \frac{\alpha+S_{n}}{\alpha+\beta+n+1}\left( \frac{{\alpha+\beta+n+1}}{\alpha+\beta+n} \right) = \\
 & = \frac{{\alpha+S_{n}}}{\alpha+\beta+n} = M_{n}
\end{align*}
$$

Pytanie: w jaki sposób pokazać że jest to proces Markova ?? 


In [ ]:
# helpers

## params
a_param = 1
b_param = 1

## helper functions
### random sample 
def polya_svm(n, a_param=a_param, b_param=b_param):
    # memory allocation
    X = np.zeros(n, dtype=int)
    S_n = 0

    for i in range(n):
        # probability of obtaining X_n = 0 (from i=0 -> n-1)
        p_i = (a_param + i - S_n) / (a_param + b_param + i)
        # we draw U ~ Unif(0, 1)
        ## case 0: we obtain X_i = 0
        if np.random.random() < p_i:
            X[i] = 0
        ## case 1: we obtain X_i =1
        else:
            S_n += 1
            X[i] = 1

    return X

In [ ]:
# part 1 - simulation
## params
n =100_0
m = 7

## simulation
experiments_part1 = list()

for _ in range(m):
    X_i = polya_svm(n)
    experiments_part1.append(X_i)

In [ ]:
# part 1 - trajectories plots 
## params
time_lim = n

## plots
for i, experiment in enumerate(experiments_part1):
    plt.plot(range(time_lim), experiment[:time_lim].cumsum() / np.ones(time_lim).cumsum(), label=f'{i} experiment')

plt.legend()
plt.show()


In [ ]:
# part 3 - simulation
## params
n = 10_00
m = 10_000
exp_configs = {
    r'Uniform distribution: $\alpha = \beta = 1$': dict(
        a_param=1.0,
        b_param=1.0
    ),
    r'Arcsine distribution $\alpha=\beta=0.5$': dict(
        a_param=0.5,
        b_param = 0.5
    ),
    r'Semi-elliptic distribution $\alpha=\beta=0.5$': dict(
        a_param=1.5,
        b_param = 1.5
    ),
    
}
## simulation

experiments_part3 = dict()
for id, config in exp_configs.items():
    experiment_values = list()
    for _ in range(m):
        X_i = polya_svm(n, **config)
        experiment_values.append(X_i)
    experiments_part3[id] = experiment_values, config

In [ ]:
# part 3 and 4 - plots

## condition loop 
for name, (experiment_values, config) in experiments_part3.items():
    ## params
    experiments_part3_plot = [experiment.cumsum()[-1]/n for experiment in experiment_values]
    domain_ex17_part3 = np.linspace(0, 1, 100)
    ## plots
    sns.histplot(experiments_part3_plot, stat='density')
    plt.title(name)
    plt.plot(domain_ex17_part3, stats.distributions.beta(*config.values()).pdf(domain_ex17_part3), color='red')
    plt.show()

### Exercise 1.18 p 44

